# **Loading Dataset**

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Download data
df = yf.download('^NDX',
                 start='2000-01-01',
                 end='2025-12-31')
# rename columns
df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']

/tmp/ipykernel_24145/2751397762.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('^NDX',
[*********************100%***********************]  1 of 1 completed


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6538 entries, 2000-01-03 to 2025-12-30
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   6538 non-null   float64
 1   High    6538 non-null   float64
 2   Low     6538 non-null   float64
 3   Open    6538 non-null   float64
 4   Volume  6538 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 306.5 KB


In [ ]:
df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2000-01-03,3790.550049,3836.860107,3643.25000,3755.739990,1510070000
2000-01-04,3546.199951,3766.570068,3542.72998,3766.570068,1511840000
2000-01-05,3507.310059,3576.169922,3371.75000,3543.129883,1735670000
2000-01-06,3340.810059,3513.550049,3334.02002,3488.310059,1598320000
2000-01-07,3529.600098,3529.750000,3314.75000,3337.260010,1634930000


In [ ]:
# Add log returns
df['Returns'] = np.log(df['Close']).diff() * 100
df.dropna(inplace=True)
df.head()

,Close,High,Low,Open,Volume,Returns
Date,,,,,,
2000-01-04,3546.199951,3766.570068,3542.729980,3766.570068,1511840000,-6.663455
2000-01-05,3507.310059,3576.169922,3371.750000,3543.129883,1735670000,-1.102722
2000-01-06,3340.810059,3513.550049,3334.020020,3488.310059,1598320000,-4.863607
2000-01-07,3529.600098,3529.750000,3314.750000,3337.260010,1634930000,5.497127
2000-01-10,3717.409912,3756.169922,3558.209961,3558.209961,1691710000,5.184259


# **Realized Volatility Definition**

In [ ]:
# define a function to calculate Yang-Zhang volatility

def yang_zhang_volatility(price_data, window=22):
    """
    Calculates Yang-Zhang Volatility.
    price_data: pd.DataFrame with ['Open', 'High', 'Low', 'Close']
    window: Lookback period (n)
      """
    log_ho = np.log(price_data['High'] / price_data['Open'])
    log_lo = np.log(price_data['Low'] / price_data['Open'])
    log_co = np.log(price_data['Close'] / price_data['Open'])

    # Overnight log-return (Open_t / Close_t-1)
    log_oc = np.log(price_data['Open'] / price_data['Close'].shift(1))

    # 1. Overnight Variance
    # We use a rolling variance of the log_oc
    sigma_overnight = log_oc.rolling(window=window).var()

    # 2. Open-to-Close Variance
    sigma_open_to_close = log_co.rolling(window=window).var()

    # 3. Rogers-Satchell Variance
    # Internal component: [h(h-c) + l(l-c)]
    rs_sum = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
    sigma_rs = rs_sum.rolling(window=window).mean()

    # 4. Calculate k (the weighting constant)
    k = 0.34 / (1.34 + (window + 1) / (window - 1))

    # 5. Final Yang-Zhang Variance
    yz_variance = sigma_overnight + k * sigma_open_to_close + (1 - k) * sigma_rs

    # Return Daily Volatility (align with GARCH (%))
    return np.sqrt(yz_variance) * 100


In [ ]:
# Calculate proxy volatility
df['YZ_Vol'] = yang_zhang_volatility(df)

# **Features Set**

## Lag RV

In [ ]:
# lag of RV
df = df.copy()
log_vol = np.log(df['YZ_Vol'])
df['log_vol_1d'] = log_vol.shift(1)
df['log_vol_5d'] = log_vol.rolling(5).mean().shift(1)
df['log_vol_22d'] = log_vol.rolling(22).mean().shift(1)

## Log VXN

In [ ]:
# download VXN ticker
vxn= yf.download("^VXN",
                  start="2000-01-01",
                  end="2025-12-31")
# rename columns
vxn.columns = ['Close', 'High', 'Low', 'Open', 'Volume']

/tmp/ipykernel_24145/4195769518.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vxn= yf.download("^VXN",
[*********************100%***********************]  1 of 1 completed


In [ ]:
vxn.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6272 entries, 2001-01-23 to 2025-12-30
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   6272 non-null   float64
 1   High    6272 non-null   float64
 2   Low     6272 non-null   float64
 3   Open    6272 non-null   float64
 4   Volume  6272 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 294.0 KB


In [ ]:
# calculate log of vxn
vxn_log = np.log(vxn['Close']).dropna()

vxn_log = vxn_log.rename('vxn_log').shift(1)

vxn_log.head()

,vxn_log
Date,
2001-01-23,NaN
2001-01-24,4.070223
2001-01-25,4.105284
2001-01-26,4.150725
2001-01-29,4.134526


In [ ]:
# concate to df
df['vxn_log'] = vxn_log

## Log Parkinson Volatility

In [ ]:
high_low_ratio = np.log(df['High'] / df['Low'])
parkinson_vol = high_low_ratio / np.sqrt(4 * np.log(2))
parkinson_vol_log = np.log(parkinson_vol).shift(1)
df['parkinson_vol_log'] = parkinson_vol_log

## 5-day EMA

In [ ]:
ema_5 = df['YZ_Vol'].ewm(span=5, adjust=False).mean()
ema_5 = ema_5.shift(1)
df['ema_5'] = ema_5

## 22-day EMA

In [ ]:
ema_22 = df['YZ_Vol'].ewm(span=22, adjust=False).mean()
ema_22 = ema_22.shift(1)
df['ema_22'] = ema_22

## Volatility of returns

In [ ]:
# 5 days
vol_5 = df["Returns"].rolling(window=5).std().shift(1)
df['vol_5'] = vol_5

# 22 days
vol_22 = df["Returns"].rolling(window=22).std().shift(1)
df['vol_22'] = vol_22

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6537 entries, 2000-01-04 to 2025-12-30
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Close              6537 non-null   float64
 1   High               6537 non-null   float64
 2   Low                6537 non-null   float64
 3   Open               6537 non-null   float64
 4   Volume             6537 non-null   int64  
 5   Returns            6537 non-null   float64
 6   YZ_Vol             6515 non-null   float64
 7   log_vol_1d         6514 non-null   float64
 8   log_vol_5d         6510 non-null   float64
 9   log_vol_22d        6493 non-null   float64
 10  vxn_log            6271 non-null   float64
 11  parkinson_vol_log  6536 non-null   float64
 12  ema_5              6514 non-null   float64
 13  ema_22             6514 non-null   float64
 14  vol_5              6532 non-null   float64
 15  vol_22             6515 non-null   float64
dtypes: flo

In [ ]:
df.head()

,Close,High,Low,Open,Volume,Returns,YZ_Vol,log_vol_1d,log_vol_5d,log_vol_22d,vxn_log,parkinson_vol_log,ema_5,ema_22,vol_5,vol_22
Date,,,,,,,,,,,,,,,,
2000-01-04,3546.199951,3766.570068,3542.729980,3766.570068,1511840000,-6.663455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,3507.310059,3576.169922,3371.750000,3543.129883,1735670000,-1.102722,NaN,NaN,NaN,NaN,NaN,-3.302402,NaN,NaN,NaN,NaN
2000-01-06,3340.810059,3513.550049,3334.020020,3488.310059,1598320000,-4.863607,NaN,NaN,NaN,NaN,NaN,-3.342476,NaN,NaN,NaN,NaN
2000-01-07,3529.600098,3529.750000,3314.750000,3337.260010,1634930000,5.497127,NaN,NaN,NaN,NaN,NaN,-3.457821,NaN,NaN,NaN,NaN
2000-01-10,3717.409912,3756.169922,3558.209961,3558.209961,1691710000,5.184259,NaN,NaN,NaN,NaN,NaN,-3.276977,NaN,NaN,NaN,NaN


In [ ]:
# download as pickle file
df.to_pickle("ndx.pkl")